# Data Story: Passantenzählung Würzburg

## Einführung

In dieser Data Story analysieren wir die Passantenfrequenz in Würzburg und vergleichen sie mit anderen bayerischen Städten. Die Analyse basiert auf öffentlich verfügbaren Daten der Passantenzählung.

**Lernziele:**
- Daten laden und aufbereiten
- Zeitreihenanalysen durchführen
- Geodaten - Visualisierungen erstellen
- Vergleichende Analysen zwischen Städten durchführen

**Hinweise für Dateninteressierte:**
- Führen alle Zellen nacheinander aus
- Achte auf die Kommentare für zusätzliche Hinweise


## Teil 1: Setup und Datenvorbereitung

Installiere und importiere alle benötigten Bibliotheken.


In [ ]:
# Install dependencies

%pip install pandas matplotlib seaborn folium numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import requests
import io
from datetime import datetime

print("Bibliotheken erfolgreich importiert")


### 1.1 Daten laden

Lade die Passantenzählungsdaten für Würzburg, Augsburg und Bamberg.


In [ ]:
WUERZBURG_URL = "https://opendata.wuerzburg.de/api/v2/catalog/datasets/passantenzaehlung_stundendaten/exports/csv?use_labels=true&limit=-1&delimiter=%3B"
AUGSBURG_URL = "https://www.augsburg.de/fileadmin/user_upload/buergerservice_rathaus/Smart_City/opendata/240429/Annastra%C3%9Fe_Frequenzdaten.csv"
BAMBERG_URL = "https://opendata.smartcitybamberg.de/Passantenfrequenzmessung/Passantenfrequenzmessung_Innenstadt_Bamberg_Rohdaten.CSV"

response = requests.get(WUERZBURG_URL)

df_wuerzburg = pd.read_csv(io.StringIO(response.text), sep=';')
df_augsburg = pd.read_csv(AUGSBURG_URL, sep=';')
df_bamberg = pd.read_csv(BAMBERG_URL, sep=';', encoding='ISO-8859-1')


### 1.2 Datenstruktur verstehen

Schauen wir uns die ersten Zeilen, sowie die Struktur der Daten an.


In [ ]:
print("=" * 120)
print("ÜBERSICHT ALLER DATASETS")
print("=" * 120)

datasets_info = {
    'Würzburg': df_wuerzburg,
    'Augsburg': df_augsburg,
    'Bamberg': df_bamberg
}

for city_name, df in datasets_info.items():
    print(f"\n{'─' * 120}")
    print(f"{city_name.upper()}")
    print(f"{'─' * 120}")
    
    print(f"\nGröße: {len(df)} Zeilen × {len(df.columns)} Spalten")
    
    print(f"\nSpalten ({len(df.columns)}):")
    for i, col in enumerate(df.columns, 1):
        print(f"   {i:2d}. {col}")
    
    print(f"\nErste 3 Zeilen:")
    display(df.head(1))

### 1.3 Datenaufbereitung

#### 1.3.1 Zeitspalten zusammenziehen für Augsburg

In [ ]:
def parse_augsburg_datetime(row):
    try:
        datum_str = row['Datum']
        stunde_str = str(row['Stunde']).strip()
        
        datum_parts = datum_str.split(',')
        if len(datum_parts) > 1:
            date_only = datum_parts[1].strip()
        else:
            date_only = datum_str.strip()
        
        hour = int(stunde_str.split()[0])
        
        datetime_obj = pd.to_datetime(date_only, format='%d.%m.%Y')
        datetime_obj = datetime_obj.replace(hour=hour)
        
        return datetime_obj
    except:
        return pd.NaT

In [ ]:
df_augsburg['Datum'] = df_augsburg.apply(parse_augsburg_datetime, axis=1)
df_augsburg.drop(columns=['Stunde'], inplace=True)

#### 1.3.2 Allgemeine Datenaufbereitung

In [ ]:
def prepare_city_data(df, city_name):
    df = df.copy()
    df['city'] = city_name
    
    date_columns = ['date', 'Zeitstempel', 'Datum', 'Zeit']
    date_col = None
    for col in date_columns:
        if col in df.columns:
            date_col = col
            break
    
    if date_col:
        df['datetime'] = pd.to_datetime(df[date_col], format='mixed')
        df['date'] = df['datetime'].dt.date
        df['year'] = df['datetime'].dt.year
        df['month'] = df['datetime'].dt.month
        df['week'] = df['datetime'].dt.isocalendar().week
        df['day_of_week'] = df['datetime'].dt.dayofweek
        df['weekday_name'] = df['datetime'].dt.day_name()
        df['hour'] = df['datetime'].dt.hour
        df['is_weekend'] = df['day_of_week'] >= 5
        if date_col != 'date':
            df.drop(columns=[date_col], inplace=True)
    
    count_columns = ['visitors', 'Passantenanzahl']
    for col in count_columns:
        if col in df.columns:
            df['Passanten'] = pd.to_numeric(df[col], errors='coerce')
            df.drop(columns=[col], inplace=True)
            break
    
    location_columns = ['Location Name', 'location', 'Standort', 'Bereich']
    for col in location_columns:
        if col in df.columns:
            df['zaehlstelle'] = df[col]
            df.drop(columns=[col], inplace=True)
            break
    
    return df

df_augsburg_prepared = prepare_city_data(df_augsburg, 'Augsburg')
df_bamberg_prepared = prepare_city_data(df_bamberg, 'Bamberg')
df_wuerzburg_prepared = prepare_city_data(df_wuerzburg, 'Würzburg')

print("Alle Städte-Daten aufbereitet")
print(f"\nWürzburg Spalten: {list(df_wuerzburg_prepared.columns)}")
print(f"Augsburg Spalten: {list(df_augsburg_prepared.columns)}")
print(f"Bamberg Spalten: {list(df_bamberg_prepared.columns)}")

---

## Teil 2: Analyse der Würzburger Passantenzählung

---


### 2.1 Karte von Würzburg mit Markierung der Zählstellen

**Fragestellung:** Wo befinden sich die Zählstellen in Würzburg?

Erstelle eine interaktive Karte mit allen Zählstellen.


In [ ]:
zahlstellen_coords = df_wuerzburg_prepared[['zaehlstelle', 'geo_point_2d']].drop_duplicates().reset_index(drop=True)

def parse_geopoint(geopoint_str):
    parts = str(geopoint_str).split(',')
    if len(parts) == 2:
        try:
            return float(parts[0].strip()), float(parts[1].strip())
        except:
            return None, None
    return None, None

zahlstellen_coords[['lat', 'lon']] = zahlstellen_coords['geo_point_2d'].apply(lambda x: pd.Series(parse_geopoint(x)))
zahlstellen_coords = zahlstellen_coords.dropna(subset=['lat', 'lon'])

print("Zählstellen in Würzburg:")
display(zahlstellen_coords[['zaehlstelle', 'lat', 'lon']])

wuerzburg_center = [zahlstellen_coords['lat'].mean(), zahlstellen_coords['lon'].mean()]

map_wuerzburg = folium.Map(
    location=wuerzburg_center,
    zoom_start=14,
    tiles='OpenStreetMap'
)

for idx, row in zahlstellen_coords.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=f"<b>{row['zaehlstelle']}</b>",
        tooltip=row['zaehlstelle'],
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(map_wuerzburg)

map_wuerzburg


### 2.2 Bewegungsmuster im Durchschnitt

**Fragestellung:** Wie viele Passanten passieren die Straßen der Zählstationen im Durchschnitt pro Jahr/Monat/Woche/Tag?

Berechne die durchschnittlichen Passantenzahlen für verschiedene Zeiträume.


In [ ]:
total_passanten = df_wuerzburg_prepared['Passanten'].sum()
unique_days = df_wuerzburg_prepared['date'].nunique()
unique_weeks = df_wuerzburg_prepared.groupby(['year', 'week']).ngroups
unique_months = df_wuerzburg_prepared.groupby(['year', 'month']).ngroups
unique_years = df_wuerzburg_prepared['year'].nunique()

avg_per_day = total_passanten / unique_days
avg_per_week = total_passanten / unique_weeks
avg_per_month = total_passanten / unique_months
avg_per_year = total_passanten / unique_years

print("=" * 80)
print("DURCHSCHNITTLICHE PASSANTENFREQUENZ IN WÜRZBURG")
print("=" * 80)
print(f"\nTotal Passanten (gesamt): {total_passanten:,.0f}")
print(f"\nDurchschnitt pro Jahr:    {avg_per_year:,.0f} Passanten")
print(f"Durchschnitt pro Monat:   {avg_per_month:,.0f} Passanten")
print(f"Durchschnitt pro Woche:   {avg_per_week:,.0f} Passanten")
print(f"Durchschnitt pro Tag:     {avg_per_day:,.0f} Passanten")


In [ ]:
df_2024 = df_wuerzburg_prepared[df_wuerzburg_prepared['year'] == 2024].copy()

weekly_data = df_2024.groupby(['week', 'zaehlstelle'])['Passanten'].sum().reset_index()
monthly_data = df_2024.groupby(['month', 'zaehlstelle'])['Passanten'].sum().reset_index()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))

for station in weekly_data['zaehlstelle'].unique():
    station_data = weekly_data[weekly_data['zaehlstelle'] == station]
    ax1.plot(station_data['week'], station_data['Passanten'], marker='o', label=station, linewidth=2)

ax1.set_xlabel('Kalenderwoche', fontsize=12)
ax1.set_ylabel('Anzahl Passanten', fontsize=12)
ax1.set_title('Passantenfrequenz pro Zählstation im Wochenverlauf 2024', fontsize=14, fontweight='bold')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

for station in monthly_data['zaehlstelle'].unique():
    station_data = monthly_data[monthly_data['zaehlstelle'] == station]
    ax2.plot(station_data['month'], station_data['Passanten'], marker='s', label=station, linewidth=2)

ax2.set_xlabel('Monat', fontsize=12)
ax2.set_ylabel('Anzahl Passanten', fontsize=12)
ax2.set_title('Passantenfrequenz pro Zählstation im Monatsverlauf 2024', fontsize=14, fontweight='bold')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(1, 13))

plt.tight_layout()
plt.show()


### 2.3 Passantenfrequenz pro Wochentag

**Fragestellung:** Wie verhält sich die Passantenfrequenz pro Zählstation auf die einzelnen Wochentage verteilt?

Analysiere die Verteilung über die Wochentage.


In [ ]:
df_wuerzburg_prepared.head(2)

In [ ]:
weekday_german = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag', 'Samstag', 'Sonntag']
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

daily_totals = df_wuerzburg_prepared.groupby(['date', 'zaehlstelle'])['Passanten'].sum().reset_index()

date_to_weekday = df_wuerzburg_prepared[['date', 'weekday_name']].drop_duplicates('date').set_index('date')['weekday_name']
daily_totals['weekday_name'] = daily_totals['date'].map(date_to_weekday)

weekday_avg = daily_totals.groupby(['weekday_name', 'zaehlstelle'])['Passanten'].mean().reset_index()
weekday_avg['weekday_order'] = pd.Categorical(
    weekday_avg['weekday_name'],
    categories=weekday_order,
    ordered=True
)
weekday_avg = weekday_avg.sort_values('weekday_order')

fig, ax = plt.subplots(figsize=(16, 8))

stations = weekday_avg['zaehlstelle'].unique()
x = np.arange(len(weekday_order))
width = 0.8 / len(stations)

for i, station in enumerate(stations):
    station_data = weekday_avg[weekday_avg['zaehlstelle'] == station]
    values = [station_data[station_data['weekday_name'] == day]['Passanten'].values[0] 
              if len(station_data[station_data['weekday_name'] == day]) > 0 else 0 
              for day in weekday_order]
    ax.bar(x + i * width, values, width, label=station, alpha=0.8)

ax.set_xlabel('Wochentag', fontsize=12)
ax.set_ylabel('Durchschnittliche Anzahl Passanten pro Tag', fontsize=12)
ax.set_title('Durchschnittliche Passantenfrequenz pro Wochentag und Zählstation', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * (len(stations) - 1) / 2)
ax.set_xticklabels(weekday_german, rotation=45)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

weekday_total = daily_totals.groupby('weekday_name')['Passanten'].mean()
print("\nDurchschnittliche Passanten pro Wochentag (alle Zählstellen):")
for day_en, day_de in zip(weekday_order, weekday_german):
    print(f"{day_de:12s}: {weekday_total[day_en]:,.0f} Passanten")

### 2.4 Passantenfrequenz im Tagesverlauf nach Wochentagen

**Fragestellung:** Wie entwickelt sich die Passantenfrequenz im Tagesverlauf, bezogen auf die jeweiligen Wochentage (aggregiert über alle Zählstationen)?

Erstelle eine Heatmap und Liniendiagramme für den Tagesverlauf.


In [ ]:
hourly_weekday = df_wuerzburg_prepared.groupby(['weekday_name', 'hour'])['Passanten'].mean().reset_index()

hourly_weekday['weekday_order'] = pd.Categorical(
    hourly_weekday['weekday_name'],
    categories=weekday_order,
    ordered=True
)
hourly_weekday = hourly_weekday.sort_values(['weekday_order', 'hour'])

pivot_hourly = hourly_weekday.pivot(index='weekday_name', columns='hour', values='Passanten')
pivot_hourly = pivot_hourly.reindex(weekday_order)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))

sns.heatmap(
    pivot_hourly,
    cmap='YlOrRd',
    annot=True,
    fmt='.0f',
    cbar_kws={'label': 'Durchschnittliche Anzahl Passanten'},
    ax=ax1,
    yticklabels=weekday_german
)
ax1.set_xlabel('Stunde des Tages', fontsize=12)
ax1.set_ylabel('Wochentag', fontsize=12)
ax1.set_title('Heatmap: Passantenfrequenz nach Wochentag und Stunde', fontsize=14, fontweight='bold')

for day_en, day_de in zip(weekday_order, weekday_german):
    day_data = hourly_weekday[hourly_weekday['weekday_name'] == day_en]
    ax2.plot(day_data['hour'], day_data['Passanten'], marker='o', label=day_de, linewidth=2)

ax2.set_xlabel('Stunde des Tages', fontsize=12)
ax2.set_ylabel('Durchschnittliche Anzahl Passanten', fontsize=12)
ax2.set_title('Passantenfrequenz im Tagesverlauf nach Wochentagen', fontsize=14, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(0, 24))

plt.tight_layout()
plt.show()


### 2.5 Passantenfrequenz pro Laufrichtung im Tagesverlauf

**Fragestellung:** Wie verhält sich die Entwicklung der Passantenfrequenz pro Laufrichtung pro Zählstation im Tagesverlauf?

Analysiere die Laufrichtungen für jede Zählstation separat.


In [ ]:
stations = df_wuerzburg_prepared['zaehlstelle'].unique()
n_stations = len(stations)

fig, axes = plt.subplots(n_stations, 1, figsize=(16, 6 * n_stations))

if n_stations == 1:
    axes = [axes]

for idx, station in enumerate(stations):
    station_data = df_wuerzburg_prepared[df_wuerzburg_prepared['zaehlstelle'] == station].copy()
    
    hourly_ltr = station_data.groupby('hour')['Passanten ltr'].mean().reset_index()
    hourly_rtl = station_data.groupby('hour')['Passanten rtl'].mean().reset_index()
    
    axes[idx].plot(hourly_ltr['hour'], hourly_ltr['Passanten ltr'], 
                    marker='o', label='Ltr (links nach rechts)', linewidth=2, color='#1f77b4')
    axes[idx].plot(hourly_rtl['hour'], hourly_rtl['Passanten rtl'], 
                    marker='s', label='Rtl (rechts nach links)', linewidth=2, color='#ff7f0e')
    
    axes[idx].set_xlabel('Stunde des Tages', fontsize=12)
    axes[idx].set_ylabel('Durchschnittliche Anzahl Passanten', fontsize=12)
    axes[idx].set_title(f'Passantenfrequenz nach Laufrichtung: {station}', 
                        fontsize=14, fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()


---

## Teil 3: Vergleich mit anderen Städten

---


### 3.1 Rangfolge der Städte

**Fragestellung:** Welche Stadt hat die höchste Passantenfrequenz in Bezug auf die meistfrequentierte Straße?

Ermittel die Rangfolge der Städte.


In [ ]:
def get_max_station_frequency(df, city_name):
    station_totals = df.groupby('zaehlstelle')['Passanten'].sum()
    max_station = station_totals.idxmax()
    max_count = station_totals.max()
    
    return {
        'Stadt': city_name,
        'Meistfrequentierte Stelle': max_station,
        'Total Passanten': max_count
    }

rankings = []
rankings.append(get_max_station_frequency(df_wuerzburg_prepared, 'Würzburg'))
rankings.append(get_max_station_frequency(df_augsburg_prepared, 'Augsburg'))
rankings.append(get_max_station_frequency(df_bamberg_prepared, 'Bamberg'))

df_rankings = pd.DataFrame(rankings)
df_rankings = df_rankings.sort_values('Total Passanten', ascending=False)
df_rankings['Rang'] = range(1, len(df_rankings) + 1)
df_rankings = df_rankings[['Rang', 'Stadt', 'Meistfrequentierte Stelle', 'Total Passanten']]

print("=" * 80)
print("RANGFOLGE DER STÄDTE NACH HÖCHSTER PASSANTENFREQUENZ")
print("=" * 80)
display(df_rankings)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(df_rankings['Stadt'], df_rankings['Total Passanten'], 
               color=['gold', 'silver', '#CD7F32'][:len(df_rankings)])

ax.set_xlabel('Total Passanten (meistfrequentierte Stelle)', fontsize=12)
ax.set_title('Städtevergleich: Höchste Passantenfrequenz', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for i, (idx, row) in enumerate(df_rankings.iterrows()):
    ax.text(row['Total Passanten'], i, f" {row['Total Passanten']:,.0f}", 
           va='center', fontweight='bold')

plt.tight_layout()
plt.show()


### 3.2 Städtevergleich: Durchschnittswerte

**Fragestellung:** Wie viele Passanten passieren die Straßen der Zählstationen der verschiedenen Städte im Durchschnitt pro Jahr/Monat/Woche/Tag?

Vergleiche die durchschnittlichen Passantenzahlen zwischen den Städten.


In [ ]:
def calculate_city_averages(df, city_name):
    total = df['Passanten'].sum()
    unique_days = df['date'].nunique()
    unique_weeks = df.groupby(['year', 'week']).ngroups if 'week' in df.columns else unique_days / 7
    unique_months = df.groupby(['year', 'month']).ngroups if 'month' in df.columns else unique_days / 30
    unique_years = df['year'].nunique() if 'year' in df.columns else unique_days / 365
    
    return {
        'Stadt': city_name,
        'Pro Tag': total / unique_days,
        'Pro Woche': total / unique_weeks,
        'Pro Monat': total / unique_months,
        'Pro Jahr': total / unique_years
    }

city_comparisons = []
city_comparisons.append(calculate_city_averages(df_wuerzburg_prepared, 'Würzburg'))
city_comparisons.append(calculate_city_averages(df_augsburg_prepared, 'Augsburg'))
city_comparisons.append(calculate_city_averages(df_bamberg_prepared, 'Bamberg'))

df_comparison = pd.DataFrame(city_comparisons)
df_comparison = df_comparison.set_index('Stadt')

print("=" * 80)
print("STÄDTEVERGLEICH: DURCHSCHNITTLICHE PASSANTENFREQUENZ")
print("=" * 80)
display(df_comparison.style.format("{:,.0f}"))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Städtevergleich: Durchschnittliche Passantenfrequenz', fontsize=16, fontweight='bold')

periods = ['Pro Tag', 'Pro Woche', 'Pro Monat', 'Pro Jahr']
colors_cities = ['#1f77b4', '#ff7f0e', '#2ca02c']

for ax, period in zip(axes.flat, periods):
    data = df_comparison[period]
    bars = ax.bar(data.index, data.values, color=colors_cities[:len(data)], alpha=0.7)
    ax.set_ylabel('Anzahl Passanten')
    ax.set_title(period, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:,.0f}',
               ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


### 3.3 Wochentagsmuster im Städtevergleich


In [ ]:
def analyze_weekday_pattern(df, city_name): 
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekday_german = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag', 'Samstag', 'Sonntag']
    
    weekday_avg = df.groupby('weekday_name')['Passanten'].mean().reindex(weekday_order)
    
    return weekday_avg, weekday_german

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Wochentagsmuster im Städtevergleich', fontsize=16, fontweight='bold')

cities_data = [
    (df_wuerzburg_prepared, 'Würzburg', axes[0]),
    (df_augsburg_prepared, 'Augsburg', axes[1]),
    (df_bamberg_prepared, 'Bamberg', axes[2])
]

for df_city, city_name, ax in cities_data:
    result = analyze_weekday_pattern(df_city, city_name)
    if result:
        weekday_avg, weekday_german = result
        ax.bar(weekday_german, weekday_avg.values, alpha=0.7)
        ax.set_title(f'{city_name}', fontweight='bold')
        ax.set_ylabel('Durchschnittliche Passanten')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


### 3.4 Tagesverlauf im Städtevergleich

Visualisiere den Tagesverlauf für alle drei Städte zum Vergleich.


In [ ]:
def get_hourly_pattern(df, city_name):
    hourly_avg = df.groupby('hour')['Passanten'].mean()
    return hourly_avg

fig, ax = plt.subplots(figsize=(16, 8))

cities_to_plot = [
    (df_wuerzburg_prepared, 'Würzburg', '#1f77b4'),
    (df_augsburg_prepared, 'Augsburg', '#ff7f0e'),
    (df_bamberg_prepared, 'Bamberg', '#2ca02c')
]

for df_city, city_name, color in cities_to_plot:
    hourly_data = get_hourly_pattern(df_city, city_name)
    if hourly_data is not None:
        ax.plot(hourly_data.index, hourly_data.values, 
               marker='o', label=city_name, linewidth=2.5, color=color)

ax.set_xlabel('Stunde des Tages', fontsize=12)
ax.set_ylabel('Durchschnittliche Anzahl Passanten', fontsize=12)
ax.set_title('Städtevergleich: Passantenfrequenz im Tagesverlauf', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 24))

plt.tight_layout()
plt.show()


---

## 4. Zusammenfassung und Erkenntnisse

---


### 4.1 Key Findings

Erstelle eine Übersicht der wichtigsten Erkenntnisse.


In [ ]:
print("=" * 80)
print("ZUSAMMENFASSUNG: WICHTIGSTE ERKENNTNISSE")
print("=" * 80)

print("\n1. RÄUMLICHE VERTEILUNG")
print(f"   - Anzahl Zählstellen Würzburg: {df_wuerzburg_prepared['zaehlstelle'].nunique()}")
print(f"   - Anzahl Zählstellen Augsburg: {df_augsburg_prepared['zaehlstelle'].nunique()}")
print(f"   - Anzahl Zählstellen Bamberg: {df_bamberg_prepared['zaehlstelle'].nunique()}")

print("\n2. ZEITLICHE MUSTER")
weekday_avg_all = df_wuerzburg_prepared.groupby('day_of_week')['Passanten'].mean()
print(f"   - Stärkster Tag: {['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So'][weekday_avg_all.idxmax()]}")
print(f"   - Schwächster Tag: {['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So'][weekday_avg_all.idxmin()]}")

hourly_avg_all = df_wuerzburg_prepared.groupby('hour')['Passanten'].mean()
print(f"   - Peak-Stunde: {hourly_avg_all.idxmax()}:00 Uhr")
print(f"   - Ruhigste Stunde: {hourly_avg_all.idxmin()}:00 Uhr")

print("\n3. STÄDTEVERGLEICH")
print(f"   - Ranking siehe oben")
print(f"   - Deutliche Unterschiede in der Passantenfrequenz, bedingt durch unterschiedliche Anzahl der Zählstellen")


---

## Ende der Data Story

**Weitere Analysemöglichkeiten:**
- Saisonale Trends über mehrere Jahre
- Einfluss von Wetter und Feiertagen
- Korrelation mit Einzelhandelsumsätzen
- Vorhersagemodelle für Passantenfrequenz
- Integration weiterer Städte

---
